# SciNoNet — Helmholtz-Potential PINN for Elastic-Wave Reconstruction

Loads the 3-ply CSV, trains the potential PINN for **spatial** and **temporal** holdout, reconstructs signals, and plots results. Run cells top to bottom.

Each training takes roughly 15-30 min on CPU (float64 second-order autograd); reduce `EPOCHS` to go faster.

## 1. Setup and imports

In [ ]:
import sys, os
sys.path.insert(0, "src")          # the scinonet package lives in ./src
import numpy as np
import torch
import matplotlib.pyplot as plt

from scinonet.seed import set_seed
from scinonet.features import SpecializedFourierFeatures
from scinonet.potential import (
    build_potential_dataset, PotentialNet, PotentialTrainConfig,
    train_potential, reconstruct_xy, evaluate_holdout,
    CP_MM_PER_S, CS_MM_PER_S,
)

set_seed(42)
torch.set_default_dtype(torch.float64)   # double precision for the PDE residual
DEVICE = torch.device("cpu")             # MPS/CUDA lack float64 double-backward

## 2. Configuration

In [ ]:
# 3-ply dataset with columns [x, y, z, t, u, v, w]
CSV = "data/dataset_1nbhd_100pts_3ply_fullsignal_6001steps.csv"
# (the plate-spanning set "data/dataset_6domain_100pts_3ply_fullsignal_6001steps.csv" also works)
SEED = 42

# Fourier-feature dispersion bands (read off the Lamb F-W plot)
K_MAX_SPATIAL, SPATIAL_SCALE, F_MAX_HZ, NUM_FREQ = 0.13, 2.0, 300e3, 160

# training
EPOCHS, BATCH = 70, 16384
SPATIAL_KEEP, N_HOLDOUT = 0.30, 8     # spatial: keep 30% of timesteps, hold out 8 whole points
TEMPORAL_KEEP          = 0.05         # temporal: keep 5% of timesteps (all points seen)

## 3. Model builder
Specialized Fourier features (z shares the in-plane band) + a tanh MLP head with fixed bulk speeds.

In [ ]:
def make_features(sc):
    L, st = sc.L, sc.s_t
    ks = K_MAX_SPATIAL * SPATIAL_SCALE
    lo = torch.tensor([-ks*L, -ks*L, -ks*L, 0.0])
    hi = torch.tensor([ ks*L,  ks*L,  ks*L, F_MAX_HZ*st])
    return SpecializedFourierFeatures(lo, hi, NUM_FREQ, seed=SEED)

def make_net(sc):
    net = PotentialNet(make_features(sc), [256, 256, 256],
                       chatp_init=sc.chat(CP_MM_PER_S), chats_init=sc.chat(CS_MM_PER_S))
    net.log_chatp.requires_grad_(False)   # fix c_p, c_s at the bulk values
    net.log_chats.requires_grad_(False)
    return net

CFG = PotentialTrainConfig(epochs=EPOCHS, batch_size=BATCH, data_only_epochs=12, balance_alpha=0.3)

## 4. Train — spatial holdout
Hold out whole points; the model must reconstruct never-seen locations.

In [ ]:
data_sp = build_potential_dataset(CSV, subsample_keep=SPATIAL_KEEP, seed=SEED, n_holdout_xy=N_HOLDOUT)
held = data_sp.holdout_xy_indices
print("held-out points:", held, "| rho =", round(data_sp.scalers.rho, 2),
      "| train rows:", len(data_sp.Xtr))

net_sp = make_net(data_sp.scalers)
hist_sp, _ = train_potential(net_sp, data_sp, CFG, DEVICE)
m_sp = evaluate_holdout(net_sp, data_sp, DEVICE, held, z=0.0)
print("\nspatial held-out median relL2 =", round(m_sp["median"], 3))

## 5. Train — temporal holdout
Keep only 5% of timesteps per point; the model must fill the gaps.

In [ ]:
data_tp = build_potential_dataset(CSV, subsample_keep=TEMPORAL_KEEP, seed=SEED)
print("train rows:", len(data_tp.Xtr))

net_tp = make_net(data_tp.scalers)
hist_tp, _ = train_potential(net_tp, data_tp, CFG, DEVICE)
m_tp = evaluate_holdout(net_tp, data_tp, DEVICE, range(data_tp.xy_points.shape[0]), z=0.0)
print("\ntemporal held-out median relL2 =", round(m_tp["median"], 3))

## 6. Prediction / reconstruction
Reconstruct the full 6001-step signal (z=0 surface) at a held-out spatial point and a seen temporal point.

In [ ]:
rec_sp = reconstruct_xy(net_sp, data_sp, held[0], 0.0, DEVICE)   # never-seen point
rec_tp = reconstruct_xy(net_tp, data_tp, 0, 0.0, DEVICE)         # seen point, gaps filled
print("reconstructed signals of shape", rec_sp["pred"].shape)

## 7. Visualization

In [ ]:
def plot_recon(rec, title, mark_seen=False):
    t = np.arange(len(rec["pred"]))
    fig, ax = plt.subplots(1, 3, figsize=(15, 3))
    for i, c in enumerate("uvw"):
        ax[i].plot(t, rec["gt"][:, i],  "k",  lw=1.0, label="ground truth")
        ax[i].plot(t, rec["pred"][:, i], "r--", lw=1.2, label="PINN")
        if mark_seen and len(rec["train_idx"]):
            ax[i].scatter(rec["train_idx"], rec["gt"][rec["train_idx"], i], s=5, c="b", label="seen")
        ax[i].set_title(f"{title} - {c}"); ax[i].set_xlabel("timestep")
    ax[0].legend(fontsize=8); plt.tight_layout(); plt.show()

plot_recon(rec_sp, "Spatial holdout (unseen point)")
plot_recon(rec_tp, "Temporal holdout (seen point)", mark_seen=True)

# loss curves (train vs validation)
for hist, name in [(hist_sp, "spatial"), (hist_tp, "temporal")]:
    plt.figure(figsize=(5, 3))
    plt.semilogy(hist["train_data"],  label="data (train)")
    plt.semilogy(hist["val_data"],    label="data (val)")
    plt.semilogy(hist["train_wave"],  label="wave")
    plt.semilogy(hist["train_gauge"], label="gauge")
    plt.semilogy(hist["train_ic"],    label="IC")
    plt.title(f"{name} losses"); plt.xlabel("epoch"); plt.legend(fontsize=8)
    plt.tight_layout(); plt.show()